# 4. Un LLM sans entraînement : Claude en *zero-shot*

**Objectif** : classer les avis avec Claude, **sans aucun entraînement**, puis le comparer à TF-IDF (93,8 %) et CamemBERT (95,8 %) sur trois critères : **précision, vitesse, coût**.

### Zero-shot, c'est quoi ?
| Approche | Ce qu'on fait | Exemples d'avis vus |
|---|---|---|
| TF-IDF + régression logistique | on entraîne un modèle de zéro | 160 000 |
| CamemBERT | on *fine-tune* un modèle pré-entraîné | 20 000 |
| **Claude zero-shot** | on **décrit la tâche en français** dans un message (le *prompt*), sans exemple | **0** |

Un LLM (*Large Language Model*) comme Claude a été entraîné sur une énorme quantité de textes : il sait déjà ce qu'est une critique de film. On lui demande simplement « cet avis est-il positif ou négatif ? ». *Zero-shot* = « zéro exemple ». (Si on lui donnait quelques exemples corrigés dans le prompt, ce serait du *few-shot*.)

**Le prix à payer** : chaque avis classé est un appel à une API payante, et prend quelques secondes. C'est exactement ce que tu vas mesurer.

Ce notebook tourne **sur ton PC** (pas besoin de GPU ni de Colab) : le calcul se fait chez Anthropic.

## 4.0 Préparer ta clé API (une seule fois)

1. Dans le dossier `avis-allocine`, crée un fichier texte nommé exactement **`.env`** contenant une seule ligne :
   ```
   ANTHROPIC_API_KEY=ta-clé-ici
   ```
2. **Ne colle jamais ta clé dans le notebook** : le notebook ira sur GitHub, et une clé publiée peut être utilisée par n'importe qui à tes frais. Le fichier `.env`, lui, est dans le `.gitignore` : git ne l'enverra jamais.

`load_dotenv()` lit ce fichier et met la clé dans les variables d'environnement, où le SDK d'Anthropic va la chercher tout seul.

💶 **Budget** : prévois quelques dollars de crédit. Tu mesureras le vrai coût sur 20 avis avant de lancer les 500.

In [ ]:
import json
import os
import time

import anthropic
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
if os.getenv("ANTHROPIC_API_KEY"):
    print("Clé trouvée ✅")
else:
    print("⚠️ Pas de clé : crée le fichier .env (voir 4.0), puis redémarre le noyau")

client = anthropic.Anthropic()

## 4.1 Les données

On ne peut pas envoyer les 20 000 avis de validation : ce serait long et cher. On tire **500 avis au hasard** (toujours les mêmes grâce à `random_state=42`).

Avec 500 avis, l'accuracy sera moins précise qu'avec 20 000 : tu calculeras cette marge d'erreur au 4.4.

In [ ]:
val = pd.read_parquet("data/validation.parquet")
echantillon = val.sample(500, random_state=42).reset_index(drop=True)

print(echantillon.shape)
print(echantillon["label"].value_counts())

## 4.2 Le prompt et le premier appel

Un appel à Claude contient :
- un **message système** (`system`) : la consigne générale, le « rôle » ;
- un **message utilisateur** : ici, l'avis à classer.

Trois réglages méritent une explication :

| Réglage | Valeur | Pourquoi |
|---|---|---|
| `output_config.format` | un schéma JSON | **sorties structurées** : Claude est *obligé* de répondre `{"sentiment": "positif"}` ou `{"sentiment": "negatif"}`. Sans ça, il pourrait répondre « Cet avis me semble plutôt positif… », difficile à exploiter automatiquement. |
| `output_config.effort` | `"low"` | Claude **réfléchit avant de répondre** (des tokens de réflexion, payés comme la sortie). La tâche est simple : on lui demande de peu réfléchir, donc moins cher et plus rapide. |
| `fallbacks` | `"default"` | si Claude refuse une demande (rare : ses filtres de sécurité peuvent se tromper), un autre modèle prend le relais automatiquement. |

**Les tokens et le coût** : comme CamemBERT, Claude découpe le texte en tokens. On paie au token, avec deux prix différents :
- tokens d'**entrée** (`input_tokens`) : le prompt + l'avis, **4 $ par million** ;
- tokens de **sortie** (`output_tokens`) : la réflexion + la réponse, **20 $ par million**.

In [ ]:
MODELE = "claude-opus-5-5"
PRIX_ENTREE = 4 / 1_000_000   # dollars par token d'entrée
PRIX_SORTIE = 20 / 1_000_000  # dollars par token de sortie

CONSIGNE = (
    "Tu classes des critiques de films en français, publiées sur Allociné. "
    "Indique si l'avis global de l'auteur sur le film est positif ou négatif. "
    "Juge le verdict de l'auteur dans son ensemble, pas des phrases isolées."
)

FORMAT = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {"sentiment": {"type": "string", "enum": ["positif", "negatif"]}},
        "required": ["sentiment"],
        "additionalProperties": False,
    },
}


def appeler_claude(texte):
    """Envoie un avis à Claude et renvoie la réponse complète de l'API."""
    return client.beta.messages.create(
        model=MODELE,
        max_tokens=2000,
        system=CONSIGNE,
        messages=[{"role": "user", "content": texte}],
        output_config={"effort": "low", "format": FORMAT},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )


avis = echantillon.loc[0, "review"]
print(avis[:300], "...")
print("Vraie réponse :", echantillon.loc[0, "label"])

reponse = appeler_claude(avis)
print()
print("stop_reason :", reponse.stop_reason)
for bloc in reponse.content:
    print(bloc.type, "→", getattr(bloc, "text", ""))
print(reponse.usage.input_tokens, "tokens d'entrée,", reponse.usage.output_tokens, "tokens de sortie")

### À TOI (4.2)
`reponse.content` est une **liste de blocs**. Il peut y avoir un bloc `thinking` (la réflexion, vide à l'affichage) puis un bloc `text` qui contient le JSON.

1. Récupère le texte du bloc dont le `type` vaut `"text"` (une boucle `for` avec un `if`, ou `next(...)`).
2. Transforme ce texte en dictionnaire Python avec `json.loads(...)`, puis récupère la valeur de `"sentiment"`.
3. Calcule le **coût de cet appel** en dollars à partir de `reponse.usage` et des deux prix.
4. Combien coûteraient **20 000 avis** à ce tarif ?

In [ ]:
# À TOI


**Mes réponses (4.2)** : ...

## 4.3 La fonction `classer` et un test sur 20 avis

Avant de dépenser de l'argent sur 500 avis, on fait un **essai pilote sur 20** : ça vérifie que tout marche et ça donne une estimation du coût total. C'est une bonne habitude en entreprise.

### À TOI (4.3)
1. Écris une fonction `classer(texte)` qui renvoie un **dictionnaire** :
   ```python
   {"prediction": 1 ou 0, "secondes": ..., "tokens_entree": ..., "tokens_sortie": ...}
   ```
   - `prediction` : 1 si `"positif"`, 0 si `"negatif"` (comme la colonne `label`) ;
   - pour mesurer le temps : `debut = time.perf_counter()` avant l'appel, puis `time.perf_counter() - debut` après ;
   - si `reponse.stop_reason == "refusal"`, mets `prediction` à `None` (Claude n'a pas répondu).
2. Applique-la aux **20 premiers avis** de `echantillon` avec une boucle, et range les résultats dans un DataFrame : `pd.DataFrame(liste_de_dictionnaires)`.
3. Affiche l'accuracy sur ces 20 avis, la latence moyenne, et le **coût total**.
4. **Estime** le coût et la durée pour 500 avis, puis pour 20 000.

*Rappel* : une proportion se calcule comme au 3.3, avec `(... == ...).mean()`.

In [ ]:
# À TOI


**Mes estimations (4.3)** : ...

## 4.4 L'évaluation sur 500 avis

Si l'estimation du 4.3 te convient, lance la boucle sur les **500 avis**. Durée probable : 15 à 30 minutes (un appel à la fois).

### À TOI (4.4)
1. Même boucle qu'au 4.3, sur tout `echantillon`. Pour voir l'avancement, remplace `range(...)` par `tqdm(range(...))` (`from tqdm.auto import tqdm`).
2. **Sauvegarde tout de suite** les résultats : `resultats.to_csv("data/claude_500.csv", index=False)`. Ainsi, si le noyau redémarre, tu relis le fichier avec `pd.read_csv(...)` au lieu de **payer une deuxième fois**.
3. Calcule l'accuracy, la latence moyenne et le coût total.
4. **La marge d'erreur** : avec seulement 500 avis, l'accuracy mesurée est approximative. Un intervalle de confiance à 95 % vaut environ $p \pm 1{,}96\sqrt{\dfrac{p(1-p)}{n}}$, avec $p$ l'accuracy et $n = 500$. Calcule-le. Claude est-il **vraiment** différent de CamemBERT (95,8 %) ?

In [ ]:
# À TOI


**Ma conclusion (4.4)** : ...

## 4.5 Les erreurs de Claude

Les chiffres ne disent pas tout. Regarde **les avis où Claude s'est trompé**.

### À TOI (4.5)
1. Ajoute les colonnes de `resultats` à `echantillon` (`pd.concat([echantillon, resultats], axis=1)`), puis filtre les lignes où `prediction != label`.
2. Lis au moins 5 de ces avis en entier. Pour chacun : est-ce **Claude qui se trompe**, ou l'**étiquette qui est discutable** (avis mitigé, note qui ne correspond pas au texte) ?

Ce travail d'analyse est très apprécié en entretien : il montre que tu ne fais pas confiance aveuglément à une métrique.

In [ ]:
# À TOI


**Mon analyse (4.5)** : ...

## 4.6 Tes avis pièges

### À TOI (4.6)
Passe tes 3 avis pièges dans `classer` et complète le tableau. (Claude donne une réponse, pas une probabilité : écris simplement sa prédiction.)

In [ ]:
# À TOI


| Avis | Vraie réponse | TF-IDF + bigrammes | CamemBERT | Claude |
|---|---|---|---|---|
| Piège 1 (« génial ») | négatif | 22 % → négatif ✅ | 0,6 % → négatif ✅ | ... |
| Piège 2 (« pas mauvais ») | positif | 38 % → négatif ❌ | 97,8 % → positif ✅ | ... |
| Piège 3 (inventé) | positif | 0,5 % → négatif ❌ | 97,8 % → positif ✅ | ... |

## 4.7 Le bilan des trois approches

C'est **le tableau central de ton projet** : il ira dans le README et tu pourras le présenter en entretien.

### À TOI (4.7)
Complète le tableau avec tes mesures. Les cases « étape 5 » seront remplies plus tard, quand on mesurera la vitesse de TF-IDF et de CamemBERT sur ton PC.

| | TF-IDF + bigrammes | CamemBERT | Claude zero-shot |
|---|---|---|---|
| Accuracy | 93,8 % (20 000 avis) | 95,8 % (20 000 avis) | ... (500 avis, ± ...) |
| Avis d'entraînement | 160 000 | 20 000 | 0 |
| Entraînement | quelques minutes, CPU | ~4 min, GPU | aucun |
| Latence par avis | étape 5 | étape 5 | ... s |
| Coût pour 1 000 avis | ~0 $ (ton PC) | ~0 $ (ton PC) | ... $ |
| Données envoyées à l'extérieur | non | non | oui (API) |

**Ma conclusion (4.7)** : quel modèle choisirais-tu pour classer **un million d'avis par jour** ? Et pour un prototype à livrer **demain**, sans données étiquetées ?